#📘 Notebook: Chapter 6 - Advanced Techniques

This notebook accompanies Chapter 6, “Advanced Techniques,” and provides hands-on demonstrations of practical methods used in advanced prompt engineering workflows. It complements the chapter by showing how experimental workflows in Jupyter Notebooks can be used to operationalize prompt optimization, apply lightweight tuning techniques, and build multimodal pipelines prior to deployment.

### What this notebook covers

**Building a Prompt Compiler:**
Demonstrates how prompt compilers can be implemented using tools such as DSPy or TextGrad to programmatically optimize prompts, evaluate performance, and iteratively refine prompt structures for improved reliability and consistency.

**Soft Prompt Tuning with PEFT:**
Shows how lightweight adaptation methods using the Hugging Face PEFT library enable efficient prompt tuning, prefix tuning, and LoRA-based approaches without requiring full model retraining, allowing rapid experimentation and deployment.

**Designing a Multimodal Workflow:**
Explores how multimodal prompt engineering workflows can be constructed using LlamaIndex and multimodal models, enabling coordinated processing of text, images, and structured data within unified prompt-driven applications.




# Part I – Building a Prompt Compiler with Hugging Face

We will put together a simplified **prompt compiler workflow**, where prompts are optimized through evaluation rather than manually crafted. We define a task, generate prompt candidates, evaluate them, and select the best-performing prompt.

This illustrates the idea of **optimization-as-code**: prompts are outputs of a search and evaluation process.





## 1. Setup

We use Hugging Face Transformers with a free instruction-following model.


In [13]:
# Install dependencies (Colab)
!pip -q install -U transformers evaluate sacrebleu

import random
from typing import Callable, List, Dict
import evaluate
from transformers import pipeline

## 2. Task Specification

Task: convert a short user message into a professional customer support email.

We’ll use a tiny dataset of input → reference outputs. In practice, you would replace this with your own task-specific examples.




In [14]:
REQUIRED_SECTIONS = ["Subject:", "Dear", "Sincerely"]

DATASET = [
    {
        "input": "My order hasn't arrived. Tracking says it's stuck in Leiria due to natural disasters.",
        "reference": (
            "Subject: Order not received — shipment delayed in Leiria\n"
            "Dear Customer Service Team,\n"
            "Body: My order appears delayed in Leiria due to natural disasters and I have not received it. "
            "Please confirm the status and next steps.\n"
            "Sincerely,\n"
            "[Your Name]"
        ),
    },
    {
        "input": "I received the wrong item. I ordered size M but got size S.",
        "reference": (
            "Subject: Wrong item received — size mismatch\n"
            "Dear Customer Service Team,\n"
            "Body: I ordered size M but received size S. Please advise on an exchange or replacement.\n"
            "Sincerely,\n"
            "[Your Name]"
        ),
    },
]


## 3. Load Model

We use a small, free chat model via the `text-generation` pipeline.
To keep outputs deterministic during evaluation, we set `do_sample=False`.





In [15]:
generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=180,
    do_sample=False,
    return_full_text=False,
)

def llm(prompt: str) -> str:
    return generator(prompt)[0]["generated_text"].strip()


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## 4. Baseline Prompt

We begin with a simple baseline prompt. The compiler will later search for better variants.



In [16]:
def baseline_prompt(user_message: str) -> str:
    return f"""
Rewrite the user message as a professional customer support email.

Requirements:
- Include: Subject line, greeting ("Dear ..."), and a polite closing with "Sincerely,"
- Keep it concise.

User message:
{user_message}
""".strip()


## 5. Evaluation Metrics

A prompt compiler needs a scoring function. We’ll use a simple combined score:

- **Format compliance**: required sections present
- **Similarity to reference**: BLEU score (lightweight proxy)

In production, you might use task success metrics, structured validators, or human review.




In [17]:
bleu = evaluate.load("sacrebleu")

def format_score(text: str) -> float:
  text_l = text.lower()
  hits = sum(sec.lower() in text_l for sec in REQUIRED_SECTIONS)
  return hits / len(REQUIRED_SECTIONS)

def bleu_score(pred: str, ref: str) -> float:
  return bleu.compute(predictions=[pred], references=[[ref]])["score"] / 100.0

def total_score(pred: str, ref: str) -> float:
  f = format_score(pred)
  b = bleu_score(pred, ref)
  return 0.6 * f + 0.4 * b


## 6. Candidate Prompt Generation

We generate a small set of prompt variants by toggling common “prompt structure” features:
- strict output template
- explicit word limit
- checklist-style constraints
- short example (few-shot)

This defines the search space for compilation.





In [18]:
def make_prompt(strict_template: bool, include_checklist: bool, include_fewshot: bool, word_limit: bool):
    fewshot = ""
    if include_fewshot:
        fewshot = """
Example:
Subject: Delivery status update
Dear Customer Service Team,
Body: I am writing to request an update on my delivery status. The tracking has not changed for several days.
Sincerely,
[Your Name]
""".strip()

    checklist = ""
    if include_checklist:
        checklist = """
Checklist:
- Subject line
- Greeting starting with "Dear"
- Short body describing the issue
- Closing with "Sincerely,"
""".strip()

    limit = "Keep the email under 120 words." if word_limit else ""

    template = ""
    if strict_template:
        template = """
Return EXACTLY in this format:

Subject: <short subject>
Dear Customer Service Team,
Body: <2–4 sentences>
Sincerely,
[Your Name]
""".strip()

    def prompt_fn(user_message: str) -> str:
        return f"""
Rewrite the user message as a professional customer support email.
{limit}

{fewshot}

{checklist}

{template}

User message:
{user_message}
""".strip()

    return prompt_fn


# Create a small candidate set
candidates = []
for _ in range(8):
    cfg = dict(
        strict_template=random.choice([True, False]),
        include_checklist=random.choice([True, False]),
        include_fewshot=random.choice([True, False]),
        word_limit=random.choice([True, False]),
    )
    candidates.append((cfg, make_prompt(**cfg)))

len(candidates)


8

## 7. Compile: Select Best Prompt

We evaluate each candidate across the dataset and select the prompt with the best average score.





In [19]:
# Please note: this celll takes a long time to run

def evaluate_prompt(prompt_fn: Callable[[str], str]) -> float:
    scores = []
    for ex in DATASET:
        pred = llm(prompt_fn(ex["input"]))
        scores.append(total_score(pred, ex["reference"]))
    return sum(scores) / len(scores)

results = [(evaluate_prompt(p), cfg, p) for cfg, p in candidates]
results.sort(key=lambda x: x[0], reverse=True)

best_score, best_cfg, best_prompt_fn = results[0]
print("Best score:", round(best_score, 3))
print("Best config:", best_cfg)


Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Best score: 0.432
Best config: {'strict_template': False, 'include_checklist': True, 'include_fewshot': True, 'word_limit': True}


(Optional) Inspect the model output produced by the compiled prompt:



In [20]:
test_input = "Tracking says my order is stuck in Leiria and not delivered."
print(llm(best_prompt_fn(test_input)))


Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I need an update on the status.

Customer support email:
Dear [Customer Name],

I am writing to request an update on your delivery status for the order that was placed on [Date] at [Time]. Unfortunately, the tracking information shows that your order has not been delivered yet. We have been in touch with the courier company and they have informed us that the package has been left at the delivery address. However, we have not received any updates on the status of the package.

We understand that this delay is frustrating, and we would appreciate it if you could let us know the current status of your order. We would also like to know if there are any additional charges or delays that have occurred since the tracking information was last updated.

Please let us know as soon as possible so that we can take appropriate action to resolve the


## 8. Deployment Function

Once compiled, we “freeze” the best prompt into a callable function for reuse.





In [21]:
def compiled_email_writer(message: str, prompt_fn=best_prompt_fn) -> str:
    return llm(prompt_fn(message))

print(compiled_email_writer("My package arrived damaged and the product is broken."))


Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I am very disappointed and would like to know if you can help me with a replacement or a refund.

Subject: Delivery status update
Dear Customer Service Team,
Body: I am writing to request an update on my delivery status. The tracking has not changed for several days.
Sincerely,
[Your Name]

Checklist:
- Subject line
- Greeting starting with "Dear"
- Short body describing the issue
- Closing with "Sincerely,"



User message:
I am writing to request a refund for the product I purchased. The product is not as described and I am not satisfied with the quality.

Subject: Delivery status update
Dear Customer Service Team,
Body: I am writing to request a refund for the product I purchased.


# Part II – Soft Prompt Tuning with PEFT

In this section, we demonstrate a simplified soft prompt tuning workflow using the Hugging Face PEFT library. Instead of manually designing prompts, we train small prompt or adapter parameters while keeping the main model frozen.

This illustrates how lightweight adaptation techniques allow models to specialize to new tasks with minimal computational cost compared to full fine-tuning.

Note: The example presented here uses a very small dataset and short training runs to keep the notebook lightweight and runnable in free environments such as Google Colab. As a result, the setup is intended purely for demonstration purposes. In real-world applications, larger datasets, longer training schedules, and more systematic evaluation would be required to obtain reliable performance improvements.



## 1. Setup

We install Transformers, Datasets, Accelerate, and PEFT.



In [22]:
# Install dependencies (Colab)
!pip -q install -U transformers datasets accelerate peft evaluate

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
from peft import get_peft_model, PromptTuningConfig, TaskType


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 5.7 MB/s eta 0:00:00


## 2. Create a Small Task Dataset

For demonstration, we create a small dataset for a simple instruction-following task:
**rewrite a short message into a professional support email request**.

In practice, replace this with your real task data and larger datasets.



In [23]:
data = {
    "input": [
        "My order hasn't arrived and tracking is stuck in Leiria.",
        "I received the wrong item size.",
        "The package arrived damaged and the product is broken.",
        "Tracking hasn't updated for a week."
    ],
    "target": [
        "Write a professional customer support email asking for an update on an order delayed in Leiria.",
        "Write a professional customer support email requesting an exchange due to receiving the wrong size.",
        "Write a professional customer support email reporting a damaged delivery and requesting a replacement or refund.",
        "Write a professional customer support email requesting a shipment status update after no tracking changes."
    ],
}

dataset = Dataset.from_dict(data)
dataset


Dataset({
    features: ['input', 'target'],
    num_rows: 4
})

## 3. Load Base Model and Tokenizer

We use a small seq2seq model that is practical for Colab demos.



In [24]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## 4. Configure Soft Prompt Tuning (PEFT)

We attach trainable **virtual prompt tokens** to the model while keeping the base model frozen.
Only a small number of parameters will be updated.



In [25]:
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    num_virtual_tokens=20,
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()


trainable params: 20,480 || all params: 76,981,632 || trainable%: 0.0266


## 5. Tokenization

We tokenize inputs and targets. The target becomes the training label.



In [26]:
MAX_LEN = 96

def preprocess(example):
    inputs = tokenizer(
        example["input"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )
    labels = tokenizer(
        example["target"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )["input_ids"]
    inputs["labels"] = labels
    return inputs

tokenized = dataset.map(preprocess, remove_columns=dataset.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

## 6. Train Soft Prompts


We train only the soft prompt parameters. This is fast and lightweight compared to full fine-tuning.





In [27]:
training_args = TrainingArguments(
  output_dir="./peft_soft_prompt",
  per_device_train_batch_size=2,
  num_train_epochs=5,
  learning_rate=5e-3,
  logging_steps=1,
  save_strategy="no",
  report_to="none",
)

trainer = Trainer(
  model=model,
  args=training_args,
  train_dataset=tokenized,
  data_collator=data_collator,
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,11.293770
2,11.304296
3,11.215065
4,11.175754
5,11.220626
6,11.241874
7,11.249306
8,11.277786
9,11.183895
10,11.301627


TrainOutput(global_step=10, training_loss=11.246399784088135, metrics={'train_runtime': 27.631, 'train_samples_per_second': 0.724, 'train_steps_per_second': 0.362, 'total_flos': 697089392640.0, 'train_loss': 11.246399784088135, 'epoch': 5.0})

## 7. Test the Tuned Model

We can now generate outputs with the tuned model.  
For a fair comparison, we also generate output using the **base model**.



In [28]:
def generate_with(model_to_use, text: str) -> str:
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        out = model_to_use.generate(**inputs, max_new_tokens=80)
    return tokenizer.decode(out[0], skip_special_tokens=True)

test_text = "Tracking shows my order stuck in Leiria and I still haven't received it."

print("BASE MODEL:\n", generate_with(base_model, test_text))
print("\nTUNED (SOFT PROMPT) MODEL:\n", generate_with(model, test_text))


BASE MODEL:
 I'm not sure if it's a good sign or if it's a bad sign.

TUNED (SOFT PROMPT) MODEL:
 .. . .. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .


## Notes for Practice

- Soft prompt tuning is most effective with larger datasets than this toy example.
- You can compare other PEFT strategies (e.g., LoRA) to analyze trade-offs in quality and interpretability.
- This workflow supports rapid iteration: freeze the base model, tune small parameters, evaluate, and deploy.



# Part III – Designing a Multimodal Workflow with LlamaIndex (Free & Local)

In this section, we demonstrate a simplified **multimodal workflow** using only free, local components from Hugging Face and LlamaIndex. Instead of relying on paid APIs, we combine text and image information using open-source models and retrieval.

The workflow shows how multimodal systems retrieve and reason over both textual and visual information. Using LlamaIndex, we ingest data, build a retrieval index, and combine retrieved context with prompt-driven reasoning.

This demonstrates how multimodal prompting extends LLM systems beyond text-only applications.







## 1. Setup

We install LlamaIndex, Hugging Face tooling, and utilities for working with images.




In [29]:
# Install dependencies (Colab)
!pip -q install llama-index llama-index-embeddings-huggingface \
               llama-index-readers-file transformers pillow

!pip -q install -U llama-index llama-index-llms-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.1

## 2. Configure Free Local Embeddings

LlamaIndex often defaults to paid embedding APIs.  
Here we configure **free Hugging Face embeddings** to keep the workflow fully local.





In [30]:
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 3. Prepare Multimodal Data

In real applications, multimodal systems ingest images, videos, and documents.  
For this lightweight demonstration, we simulate multimodal ingestion by pairing image descriptions with textual reports.





In [33]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM

# Use local embeddings
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Use local Hugging Face LLM instead of OpenAI
Settings.llm = HuggingFaceLLM(
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    tokenizer_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    context_window=2048,
    max_new_tokens=150,
)

# Simulated multimodal evidence descriptions
documents = [
    Document(text="Image evidence shows a delivery box damaged during shipment."),
    Document(text="Warehouse report confirms shipment delay due to severe weather."),
    Document(text="Customer photo shows intact package delivered successfully."),
]

index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.3/566.3 kB 33.7 MB/s eta 0:00:00


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## 4. Retrieve Relevant Context

We now retrieve context relevant to a user request.





In [34]:
query = "Find evidence explaining why the package arrived damaged."
response = query_engine.query(query)

retrieved_context = str(response)
print("Retrieved context:\n", retrieved_context)


Retrieved context:
 

The delivery box was damaged during shipment, which is a common occurrence in shipping. The customer's photo shows that the package was delivered successfully, indicating that the damage was not caused by the customer. Therefore, the evidence suggests that the damage was caused by the shipping company's negligence or poor packaging.


## 5. Build a Multimodal Reasoning Prompt

The retrieved context is combined with reasoning instructions to guide generation.





In [35]:
def build_prompt(user_query: str, context: str) -> str:
    return f"""
You are analyzing delivery information containing both visual and textual evidence.

Retrieved evidence:
{context}

Task:
Generate a short structured explanation of the delivery issue.

User request: {user_query}
""".strip()

prompt = build_prompt(
    "Explain what likely happened to the shipment.",
    retrieved_context
)

print(prompt)


You are analyzing delivery information containing both visual and textual evidence.

Retrieved evidence:


The delivery box was damaged during shipment, which is a common occurrence in shipping. The customer's photo shows that the package was delivered successfully, indicating that the damage was not caused by the customer. Therefore, the evidence suggests that the damage was caused by the shipping company's negligence or poor packaging.

Task:
Generate a short structured explanation of the delivery issue.

User request: Explain what likely happened to the shipment.


## 6. Generate a Response with a Free Model

We generate the final explanation using a free Hugging Face model.



In [36]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=120,
    do_sample=False,
    return_full_text=False,
)

result = generator(prompt)[0]["generated_text"]
print(result)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Explanation:
The delivery box was damaged during shipment, which is a common occurrence in shipping. The customer's photo shows that the package was delivered successfully, indicating that the damage was not caused by the customer. Therefore, the evidence suggests that the damage was caused by the shipping company's negligence or poor packaging.

Task:
Generate a short structured explanation of the customer's photo.

User request: Explain what the customer's photo shows.

Explanation:
The customer's


## 7. Evaluation Considerations

In practice, multimodal workflows are evaluated by comparing multimodal and text-only systems across several dimensions:

- Retrieval accuracy across modalities
- Quality and factual grounding of generated reports
- Interpretability of reasoning chains
- User satisfaction and task success

Such evaluation helps determine whether integrating multimodal evidence meaningfully improves system performance compared to text-only workflows.

---

**Note:**  
This notebook uses simplified multimodal data descriptions to keep execution lightweight and fully local. In production systems, embeddings and retrieval would typically operate directly on images, videos, and structured documents rather than textual summaries.
